# Cross-Species Conservation Pattern Analysis
**Data**: `/api/v1/export/conservation`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
from matplotlib_venn import venn3
import warnings
warnings.filterwarnings('ignore')
sns.set_palette('Set2')
SPECIES_MAP = {'人类': 'Human', '黑猩猩': 'Chimpanzee', '猕猴': 'Macaque', '狨猴': 'Marmoset'}
API_BASE_URL = 'http://localhost:8000/api/v1'
print('Setup complete')

In [ ]:
response = requests.get(f'{API_BASE_URL}/export/conservation', params={'min_species_count': 2, 'limit': 5000})
df = pd.DataFrame(response.json()['data'])
print(f'Fetched {len(df)} conserved lncRNAs')
df.head()

## Conservation Statistics

In [ ]:
summary = df.groupby('species_count').agg({'core_id':'count', 'total_regulations':'sum', 'avg_binding_affinity':'mean'}).reset_index()
summary.columns = ['Species Count', 'lncRNA Count', 'Total Regulations', 'Avg BA']
print(summary)
summary.to_excel('results/conservation_summary.xlsx', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie(summary['lncRNA Count'], labels=[f"{int(r['Species Count'])} Species" for _,r in summary.iterrows()], autopct='%1.1f%%')
axes[0].set_title('Conservation Distribution', fontweight='bold')
axes[1].bar(summary['Species Count'].astype(str)+' Species', summary['Total Regulations'], color=sns.color_palette('Set2'), edgecolor='black')
axes[1].set_title('Regulations by Conservation', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/05_conservation_distribution.png', dpi=300)
plt.show()

## Species Sharing Matrix

In [ ]:
def extract_species(names):
    s = set()
    for n in names:
        if '_chimp' in n: s.add('Chimpanzee')
        elif '_macaque' in n: s.add('Macaque')
        elif '_marmoset' in n: s.add('Marmoset')
        else: s.add('Human')
    return s
df['species_set'] = df['lncrna_names'].apply(extract_species)
species = ['Human', 'Chimpanzee', 'Macaque', 'Marmoset']
matrix = np.zeros((4,4), dtype=int)
for i,s1 in enumerate(species):
    for j,s2 in enumerate(species):
        if i==j: matrix[i,j] = sum(s1 in ss for ss in df['species_set'])
        else: matrix[i,j] = sum((s1 in ss and s2 in ss) for ss in df['species_set'])
shared_df = pd.DataFrame(matrix, index=species, columns=species)
print(shared_df)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(shared_df, annot=True, fmt='d', cmap='YlOrRd', square=True, linewidths=1)
plt.title('Cross-Species Conservation Matrix', fontweight='bold')
plt.savefig('figures/06_conservation_heatmap.png', dpi=300)
plt.show()

## Statistical Tests

In [ ]:
groups = [df[df['species_count']==sc]['total_regulations'].values for sc in [2,3,4]]
h_stat, p_val = stats.kruskal(*groups)
print(f'Kruskal-Wallis: H={h_stat:.4f}, p={p_val:.4e}')
corr, p_corr = stats.spearmanr(df['species_count'], df['total_regulations'])
print(f'Spearman: rho={corr:.4f}, p={p_corr:.4e}')

In [ ]:
print('KEY FINDINGS')
for _,r in summary.iterrows():
    print(f'{int(r["Species Count"])} species: {int(r["lncRNA Count"])} lncRNAs')